In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Informações Diárias 

In [0]:
bronze_path_cvm = "/Volumes/workspace/case_spark_cvm/bronze/cvm_informe_diario/"

df_silver_cvm = ler_ultima_particao_delta(spark, bronze_path_cvm)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_cvm = df_silver_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_silver_cvm = df_silver_cvm.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['TP_FUNDO_CLASSE', 'CNPJ_FUNDO_CLASSE', 'VL_TOTAL']

# Aplicando a filtro para dropar as colunas
df_silver_cvm = df_silver_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Retirando dados duplicados

Com a mudança de rosolução da CVM (***Resolução CVM 175***), Com a nova regra, os fundos passaram a ser estruturados em classes e subclasses, adotando o tipo "CLASSES - FIF" (Fundo de Investimento Financeiro).
Caso acha dados do mesmo ***CNPJ_FUNDO_CLASSE***, os dados de "CLASSES - FIF" terão prioridade e o evento com nomecclatura antiga será excluido.


```
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|TP_FUNDO_CLASSE| CNPJ_FUNDO_CLASSE|ID_SUBCLASSE| DT_COMPTC|   VL_TOTAL|      VL_QUOTA|VL_PATRIM_LIQ|CAPTC_DIA|RESG_DIA|NR_COTST|data_processamento|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
|  CLASSES - FIF|12.586.174/0001-67|        NULL|2026-01-07|47445220.02|1.406455790000|  47448983.02|     0.00|    0.00|       1|          20260221|
|             FI|12.586.174/0001-67|        NULL|2026-01-07|47445414.65|1.406474270000|  47449606.58|     0.00|    0.00|       1|          20260221|
+---------------+------------------+------------+----------+-----------+--------------+-------------+---------+--------+--------+------------------+
```

In [0]:
# Coluna temporaria para definir prioridade em CLASSES - FIF
df_silver_cvm = df_silver_cvm.withColumn(
    "prioridade_tipo",
    f.when(f.col("TP_FUNDO_CLASSE") ==  "CLASSES - FIF", 1).otherwise(2)
)

# Definindo a janela  particionando pelas colunas CORE
window_spec = Window.partitionBy("CNPJ_FUNDO_CLASSE", "ID_SUBCLASSE", "DT_COMPTC").orderBy("prioridade_tipo")

#Aplicamos a numeração das linhas (row_number) dentro de cada janela
df_silver_cvm = df_silver_cvm.withColumn("row_num", f.row_number().over(window_spec))

# filtrando prioridade_tipo = 1 de cada grupo e removendo as colunas auxiliares 
df_silver_cvm = df_silver_cvm.filter(f.col("row_num") == 1).drop("prioridade_tipo", "row_num")


#### 1.1.4 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze e Colunas não serão utilizadas
df_silver_cvm = df_silver_cvm.drop("TP_FUNDO", "CNPJ_FUNDO", "data_processamento")


# Criando a Data de Processamento da silver
df_silver_cvm = df_silver_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
df_silver_cvm = df_silver_cvm\
    .withColumn('tp_fundo_classe', f.col('TP_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType()))\
    .withColumn('id_subclasse', f.col('ID_SUBCLASSE').cast(t.StringType()))\
    .withColumn('dt_comptc', f.col('DT_COMPTC').cast(t.DateType()))\
    .withColumn('vl_total', f.col('VL_TOTAL').cast(t.DecimalType(38,2)))\
    .withColumn('vl_quota', f.col('VL_QUOTA').cast(t.DecimalType(38,11)))\
    .withColumn('vl_patrim_liq', f.col('VL_PATRIM_LIQ').cast(t.DecimalType(38,2)))\
    .withColumn('captc_dia', f.col('CAPTC_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('resg_dia', f.col('RESG_DIA').cast(t.DecimalType(38,2)))\
    .withColumn('nr_cotst', f.col('NR_COTST').cast(t.LongType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))\

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_silver_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fundos_diario")